# VoxelMorph – Distanz **GT-Lesion ↔ Pred-Lesion** (im Needle-Bild / fixed-space)

Dieses Notebook ist für deinen **VXM-Workflow** gedacht:

- **moving** = Initialaufnahme
- **fixed** = Needle-Aufnahme
- **Flow** kommt aus der VoxelMorph-Inference und liegt auf dem **fixed/Needle-Gitter**
- die **Lesion** aus der **Initialaufnahme** wird in den **Needle-Raum** propagiert
- die **GT-Lesion im Needle-Bild** wird direkt aus der **Needle-Annotation** geladen
- anschließend wird die Distanz zwischen **GT lesion** und **predicted lesion** in **mm** berechnet

## Wichtige Annahme zur Flow-Konvention

Dieses Notebook nimmt standardmäßig an, dass der gespeicherte Flow ein **backward map auf dem fixed-Gitter** ist:

\[
x_{moving} = x_{fixed} + flow(x_{fixed})
\]

Gesucht ist der Punkt im fixed/needle-Raum zu einem gegebenen Punkt im moving/initial-Raum.  
Dafür wird numerisch invertiert:

\[
x_f^{k+1} = x_m - flow(x_f^k)
\]

Das passt zur üblichen VoxelMorph-Interpretation für `moving -> fixed`, wenn `warp = model(moving, fixed)` erzeugt wird.

## Ergebnis

Pro Fall bekommst du:

- `xyz_lesion_gt_mm` = GT-Läsionsposition im Needle-Raum
- `xyz_lesion_pred_mm` = vorhergesagte Läsionsposition im Needle-Raum
- `dist_gt_lesion_pred_lesion_mm` = 3D-Distanz in mm

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

plt.rcParams.update({"figure.figsize": (6, 6), "axes.grid": False})
np.set_printoptions(suppress=True, precision=3)

In [2]:
# ============================================================
# Basispfade
# ============================================================

VXM_ROOT = Path("/home/students/studunals1/biopsy_project/vxm").resolve()
CEPH_ROOT = Path("/mnt/ceph/vol_02_home_students/studunals1/biopsy_project").resolve()

NOTEBOOK_DIR = (VXM_ROOT / "distances").resolve()
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_TEST_CSV = (VXM_ROOT / "Analysis" / "test_lesion_0.csv").resolve()

DEFAULT_INF_BASE_DIR_CANDS = [
    (VXM_ROOT / "inference").resolve(),
    (VXM_ROOT / "ncc" / "lesion" / "inference").resolve(),
    (VXM_ROOT / "results" / "inference").resolve(),
]

DEFAULT_INF_BASE_DIR = None
for c in DEFAULT_INF_BASE_DIR_CANDS:
    if c.exists():
        DEFAULT_INF_BASE_DIR = c
        break
if DEFAULT_INF_BASE_DIR is None:
    DEFAULT_INF_BASE_DIR = DEFAULT_INF_BASE_DIR_CANDS[0]

print("VXM_ROOT         :", VXM_ROOT)
print("CEPH_ROOT        :", CEPH_ROOT)
print("NOTEBOOK_DIR     :", NOTEBOOK_DIR)
print("DEFAULT_TEST_CSV :", DEFAULT_TEST_CSV)
print("DEFAULT_INF_BASE :", DEFAULT_INF_BASE_DIR)

VXM_ROOT         : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm
CEPH_ROOT        : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project
NOTEBOOK_DIR     : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/distances
DEFAULT_TEST_CSV : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/Analysis/test_lesion_0.csv
DEFAULT_INF_BASE : /mnt/ceph/vol_02_home_students/studunals1/biopsy_project/vxm/inference


In [3]:
# ============================================================
# Helper: Pfade + CSV
# ============================================================

def resolve_existing_path(p_str: str, extra_roots=None) -> Path:
    if p_str is None or (isinstance(p_str, float) and np.isnan(p_str)):
        raise FileNotFoundError(f"Pfad ist leer/NaN: {p_str}")

    p = Path(str(p_str)).expanduser()

    if p.is_absolute() and p.exists():
        return p.resolve()

    roots = [VXM_ROOT]
    if extra_roots:
        roots += [Path(r) for r in extra_roots]

    for r in roots:
        cand = (r / p).resolve()
        if cand.exists():
            return cand

    raise FileNotFoundError(f"Pfad nicht gefunden: {p_str} (probiert roots={roots})")


def load_pairs_csv(csv_path: Path) -> pd.DataFrame:
    csv_path = Path(csv_path)
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV nicht gefunden: {csv_path}")

    df_ = pd.read_csv(csv_path)
    required = ["moving", "fixed", "patient_id", "study_id", "breast_side"]
    missing = [c for c in required if c not in df_.columns]
    if missing:
        raise ValueError(
            f"CSV hat nicht alle benötigten Spalten.\n"
            f"Fehlend: {missing}\n"
            f"Vorhanden: {list(df_.columns)}\n"
            f"CSV: {csv_path}"
        )
    return df_

In [4]:
# ============================================================
# Annotationen laden / finden
# ============================================================

CANDIDATE_LABELS_LESION = ["Lesion", "Läsion", "Tumor"]

def load_points_from_json(json_path: Path):
    json_path = Path(json_path)
    with json_path.open("r") as f:
        data = json.load(f)

    annotations = data.get("annotations", [])
    points = []

    for ann in annotations:
        for p in ann.get("points", []):
            lbl = p.get("name") or p.get("label")
            coords = p.get("coords_transformed") or p.get("coords")
            if lbl is None or coords is None or len(coords) < 3:
                continue
            points.append({"label": lbl, "xyz_mm": np.array(coords[:3], dtype=float)})

    return points


def _json_has_label(json_path: Path, candidate_labels):
    try:
        pts = load_points_from_json(json_path)
    except Exception:
        return False

    cand = {c.lower() for c in candidate_labels}
    for p in pts:
        lbl = (p.get("label") or "").lower()
        if lbl in cand:
            return True
    return False


def get_point_for_labels(json_path: Path, candidate_labels):
    pts = load_points_from_json(json_path)
    cand = {c.lower() for c in candidate_labels}

    matches = [p for p in pts if (p.get("label") or "").lower() in cand]
    if not matches:
        raise RuntimeError(
            f"Keine Annotation mit Labels {candidate_labels} in {json_path.name}. "
            f"Verfügbare Labels: {[p['label'] for p in pts]}"
        )

    if len(matches) > 1:
        print(f"[WARN] Mehrere passende Punkte in {json_path.name}; nehme den ersten: "
              f"{[m['label'] for m in matches]}")

    return matches[0]


def find_annotations_dir_from_volume(vol_path: Path, max_up=8) -> Path:
    vol_path = Path(vol_path).resolve()
    cur = vol_path.parent
    for _ in range(max_up):
        cand = cur / "annotations"
        if cand.is_dir():
            return cand
        cur = cur.parent
    raise FileNotFoundError(f"Kein annotations/-Ordner in Elternpfaden von {vol_path}")


def find_lesion_json_for_moving(anno_dir: Path, candidate_labels=CANDIDATE_LABELS_LESION):
    '''
    moving Lesion-Annotation (Initial/KMDyn):
    - bevorzugt 'kmdyn' oder 'init/initial'
    - penalisiert 'needle'
    '''
    anno_dir = Path(anno_dir)
    if not anno_dir.is_dir():
        raise FileNotFoundError(f"annotations/-Ordner nicht gefunden: {anno_dir}")

    anno_files = sorted([p for p in anno_dir.iterdir() if p.is_file() and p.name.lower().endswith(".json")])
    if not anno_files:
        raise FileNotFoundError(f"Keine .json-Annotationen in {anno_dir}")

    candidates = []
    for f in anno_files:
        if not _json_has_label(f, candidate_labels):
            continue
        name_l = f.name.lower()
        score = 0
        if "kmdyn" in name_l:
            score += 5
        if "init" in name_l or "initial" in name_l:
            score += 4
        if "needle" in name_l or "nadel" in name_l:
            score -= 3
        if "marker" in name_l:
            score -= 2
        candidates.append((score, f))

    if not candidates:
        raise RuntimeError(f"Keine passende moving-Lesion-Annotation in {anno_dir} gefunden.")

    candidates.sort(key=lambda t: t[0], reverse=True)
    best_score, best_file = candidates[0]
    print(f"[INFO] moving Lesion-Annotation: {best_file.name} (Score={best_score})")
    return best_file


def find_lesion_json_for_needle(anno_dir: Path, candidate_labels=CANDIDATE_LABELS_LESION):
    '''
    Needle Lesion-Annotation:
    - bevorzugt 'needle'/'nadel'
    - penalisiert 'kmdyn'/'init'
    '''
    anno_dir = Path(anno_dir)
    if not anno_dir.is_dir():
        raise FileNotFoundError(f"annotations/-Ordner nicht gefunden: {anno_dir}")

    anno_files = sorted([p for p in anno_dir.iterdir() if p.is_file() and p.name.lower().endswith(".json")])
    if not anno_files:
        raise FileNotFoundError(f"Keine .json-Annotationen in {anno_dir}")

    candidates = []
    for f in anno_files:
        if not _json_has_label(f, candidate_labels):
            continue
        name_l = f.name.lower()
        score = 0
        if "needle" in name_l or "nadel" in name_l:
            score += 5
        if "kmdyn" in name_l:
            score -= 2
        if "init" in name_l or "initial" in name_l:
            score -= 2
        if "marker" in name_l:
            score -= 2
        candidates.append((score, f))

    if not candidates:
        raise RuntimeError(f"Keine passende Needle-Lesion-Annotation in {anno_dir} gefunden.")

    candidates.sort(key=lambda t: t[0], reverse=True)
    best_score, best_file = candidates[0]
    print(f"[INFO] needle Lesion-Annotation: {best_file.name} (Score={best_score})")
    return best_file

In [5]:
# ============================================================
# Inference-Run / Session Discovery (VXM-kompatibel)
# ============================================================

def list_session_dirs(run_dir: Path):
    run_dir = Path(run_dir)
    if not run_dir.is_dir():
        return []
    return sorted([d for d in run_dir.iterdir() if d.is_dir() and d.name.startswith("inf_")])


def discover_runs(base_dir: Path):
    '''
    Unterstützt zwei Layouts:
    1) base_dir = .../inference/lesion_ncc_xyz     und darin inf_*
    2) base_dir = .../inference                     und darin mehrere run-Ordner
    '''
    base_dir = Path(base_dir)
    if not base_dir.is_dir():
        return []

    children = sorted([d for d in base_dir.iterdir() if d.is_dir()])
    if any(d.name.startswith("inf_") for d in children):
        return [base_dir]

    return children


def pick_latest_session_dir(run_dir: Path) -> Path:
    run_dir = Path(run_dir)
    if run_dir.name.startswith("inf_"):
        return run_dir

    sess = list_session_dirs(run_dir)
    if sess:
        return sess[-1]

    # Falls run_dir direkt Case-Ordner enthält, nimm run_dir
    return run_dir


def load_flow_for_case(patient_id, study_id, breast_side, run_dir: Path, session_dir: Path = None):
    if session_dir is None:
        session_dir = pick_latest_session_dir(run_dir)

    case_dir = Path(session_dir) / f"{patient_id}_{study_id}_{breast_side}"
    flow_path = case_dir / "flow.nii.gz"
    if not flow_path.exists():
        raise FileNotFoundError(f"flow.nii.gz nicht gefunden für Fall: {case_dir}")

    flow_img = nib.load(str(flow_path))
    flow_np = flow_img.get_fdata().astype(np.float32)

    # erwartet gespeichert als (X, Y, Z, 3); wandle zu (3, X, Y, Z)
    if flow_np.ndim != 4 or flow_np.shape[-1] != 3:
        raise ValueError(f"Unerwartete Flow-Shape {flow_np.shape} in {flow_path}")

    flow_np = np.moveaxis(flow_np, -1, 0)
    return flow_np, flow_path

In [6]:
# ============================================================
# Punkt-Mapping: moving -> fixed via Inversion
# ============================================================

def _sample_flow_trilinear(flow_np: np.ndarray, x: float, y: float, z: float) -> np.ndarray:
    '''
    Trilineares Sampling eines Flows.
    flow_np: (3, X, Y, Z) auf FIXED-Gitter
    '''
    _, X, Y, Z = flow_np.shape

    x = float(np.clip(x, 0, X - 1))
    y = float(np.clip(y, 0, Y - 1))
    z = float(np.clip(z, 0, Z - 1))

    x0 = int(np.floor(x)); x1 = min(x0 + 1, X - 1)
    y0 = int(np.floor(y)); y1 = min(y0 + 1, Y - 1)
    z0 = int(np.floor(z)); z1 = min(z0 + 1, Z - 1)

    xd = x - x0
    yd = y - y0
    zd = z - z0

    def f(ix, iy, iz):
        return flow_np[:, ix, iy, iz]

    c000 = f(x0, y0, z0); c100 = f(x1, y0, z0)
    c010 = f(x0, y1, z0); c110 = f(x1, y1, z0)
    c001 = f(x0, y0, z1); c101 = f(x1, y0, z1)
    c011 = f(x0, y1, z1); c111 = f(x1, y1, z1)

    c00 = c000 * (1 - xd) + c100 * xd
    c10 = c010 * (1 - xd) + c110 * xd
    c01 = c001 * (1 - xd) + c101 * xd
    c11 = c011 * (1 - xd) + c111 * xd

    c0 = c00 * (1 - yd) + c10 * yd
    c1 = c01 * (1 - yd) + c11 * yd

    return (c0 * (1 - zd) + c1 * zd).astype(np.float32)


def moving_point_mm_to_fixed_mm_via_inversion(
    point_xyz_mm_moving,
    moving_pre_path,
    fixed_pre_path,
    flow_np_fixedgrid,
    n_iter: int = 20,
):
    moving_pre_path = Path(moving_pre_path)
    fixed_pre_path  = Path(fixed_pre_path)

    mv_img = nib.load(str(moving_pre_path))
    fx_img = nib.load(str(fixed_pre_path))

    mv_aff = mv_img.affine
    fx_aff = fx_img.affine

    mv_aff_inv = np.linalg.inv(mv_aff)
    fx_aff_inv = np.linalg.inv(fx_aff)

    x_m_vox = np.array(nib.affines.apply_affine(mv_aff_inv, np.array(point_xyz_mm_moving, dtype=float)), dtype=np.float32)

    # Startschätzer: gleiche Weltkoordinate direkt in fixed-Voxel
    x_f_vox = np.array(nib.affines.apply_affine(fx_aff_inv, np.array(point_xyz_mm_moving, dtype=float)), dtype=np.float32)

    history = [x_f_vox.copy()]
    for _ in range(int(n_iter)):
        disp = _sample_flow_trilinear(flow_np_fixedgrid, x_f_vox[0], x_f_vox[1], x_f_vox[2])
        x_f_vox = x_m_vox - disp
        history.append(x_f_vox.copy())

    x_f_mm = np.array(nib.affines.apply_affine(fx_aff, x_f_vox), dtype=np.float32)
    dbg = {
        "x_m_vox": x_m_vox,
        "x_f_vox_final": x_f_vox,
        "iter_history_vox": np.array(history, dtype=np.float32),
    }
    return x_f_mm, dbg

In [7]:
# ============================================================
# Visualisierung (volle Needle-Slice, kein Crop-Zoom)
# ============================================================

def mm_to_vox(img: nib.Nifti1Image, xyz_mm):
    aff_inv = np.linalg.inv(img.affine)
    return np.array(nib.affines.apply_affine(aff_inv, np.array(xyz_mm, dtype=float)), dtype=float)

def show_points_on_fixed_volume(
    fixed_path,
    xyz_pred_mm,
    xyz_gt_mm,
    axis="z",
    patch=41,
    title="",
    dist_mm=None,
):
    fixed_path = Path(fixed_path)
    img = nib.load(str(fixed_path))
    vol = img.get_fdata().astype(np.float32)

    p_pred = mm_to_vox(img, xyz_pred_mm)
    p_gt = mm_to_vox(img, xyz_gt_mm)

    if axis == "z":
        sl = int(np.clip(round((p_pred[2] + p_gt[2]) / 2), 0, vol.shape[2] - 1))
        img2d = vol[:, :, sl]
        x_pred, y_pred = p_pred[0], p_pred[1]
        x_gt, y_gt = p_gt[0], p_gt[1]
    elif axis == "y":
        sl = int(np.clip(round((p_pred[1] + p_gt[1]) / 2), 0, vol.shape[1] - 1))
        img2d = vol[:, sl, :]
        x_pred, y_pred = p_pred[0], p_pred[2]
        x_gt, y_gt = p_gt[0], p_gt[2]
    elif axis == "x":
        sl = int(np.clip(round((p_pred[0] + p_gt[0]) / 2), 0, vol.shape[0] - 1))
        img2d = vol[sl, :, :]
        x_pred, y_pred = p_pred[1], p_pred[2]
        x_gt, y_gt = p_gt[1], p_gt[2]
    else:
        raise ValueError("axis muss x, y oder z sein")

    plt.figure(figsize=(6.5, 6.5))
    plt.imshow(img2d.T, cmap="gray", origin="lower", interpolation="none")
    plt.scatter([x_gt], [y_gt], marker="o", s=80, facecolors="none", edgecolors="red", linewidths=2, label="GT Lesion")
    plt.scatter([x_pred], [y_pred], marker="x", s=100, c="deepskyblue", linewidths=2, label="Pred Lesion")

    if patch and patch > 1:
        half = patch // 2
        cx = (x_pred + x_gt) / 2
        cy = (y_pred + y_gt) / 2
        rect = plt.Rectangle(
            (cx - half, cy - half),
            patch, patch,
            fill=False,
            linestyle="--",
            linewidth=1.2,
            edgecolor="cyan"
        )
        plt.gca().add_patch(rect)

    ttl = title or f"GT Lesion & Pred Lesion in Needle | Slice {axis}={sl}"
    if dist_mm is not None and np.isfinite(dist_mm):
        ttl += f" | Dist={dist_mm:.2f} mm"
    plt.title(ttl)
    plt.xlabel("x")
    plt.ylabel("y")
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [8]:
# ============================================================
# Single Case / Alle Fälle
# ============================================================

def compute_gt_lesion_pred_lesion_for_case(
    df: pd.DataFrame,
    case_idx: int,
    run_dir: Path,
    session_dir: Path = None,
    inv_iters: int = 20,
    verbose: bool = True,
):
    if not (0 <= case_idx < len(df)):
        raise IndexError(f"case_idx {case_idx} außerhalb 0..{len(df)-1}")

    row = df.iloc[case_idx]
    pid   = row["patient_id"]
    study = row["study_id"]
    side  = row["breast_side"]

    moving_initial_pre = resolve_existing_path(row["moving"], extra_roots=[CEPH_ROOT])
    fixed_needle_pre   = resolve_existing_path(row["fixed"],  extra_roots=[CEPH_ROOT])

    if verbose:
        print(f"\n=== Fall {case_idx:02d} ===")
        print(f"patient_id : {pid}")
        print(f"study_id   : {study}")
        print(f"breast_side: {side}")
        print("moving_initial_pre:", moving_initial_pre)
        print("fixed_needle_pre  :", fixed_needle_pre)

    anno_dir = find_annotations_dir_from_volume(fixed_needle_pre)
    if verbose:
        print("Anno-Dir:", anno_dir)

    lesion_json_moving = find_lesion_json_for_moving(anno_dir)
    lesion_info_moving = get_point_for_labels(lesion_json_moving, CANDIDATE_LABELS_LESION)
    xyz_lesion_moving_mm = lesion_info_moving["xyz_mm"]
    if verbose:
        print("Lesion-moving-Info:", lesion_info_moving)

    lesion_json_needle = find_lesion_json_for_needle(anno_dir)
    lesion_info_needle = get_point_for_labels(lesion_json_needle, CANDIDATE_LABELS_LESION)
    xyz_lesion_gt_mm = lesion_info_needle["xyz_mm"]
    if verbose:
        print("Lesion-needle-Info:", lesion_info_needle)

    flow_np_fixedgrid, flow_path = load_flow_for_case(pid, study, side, run_dir=run_dir, session_dir=session_dir)

    xyz_lesion_pred_mm, dbg = moving_point_mm_to_fixed_mm_via_inversion(
        point_xyz_mm_moving=xyz_lesion_moving_mm,
        moving_pre_path=moving_initial_pre,
        fixed_pre_path=fixed_needle_pre,
        flow_np_fixedgrid=flow_np_fixedgrid,
        n_iter=inv_iters,
    )

    dist_mm = float(np.linalg.norm(xyz_lesion_gt_mm - xyz_lesion_pred_mm))

    res = {
        "patient_id": pid,
        "study_id": study,
        "breast_side": side,
        "moving_initial_pre_path": str(moving_initial_pre),
        "fixed_needle_pre_path": str(fixed_needle_pre),
        "flow_path": str(flow_path),
        "xyz_lesion_moving_mm": xyz_lesion_moving_mm,
        "xyz_lesion_gt_mm": xyz_lesion_gt_mm,
        "xyz_lesion_pred_mm": xyz_lesion_pred_mm,
        "dist_gt_lesion_pred_lesion_mm": dist_mm,
        "dbg": dbg,
    }
    return res


def run_all_cases(
    df: pd.DataFrame,
    run_dir: Path,
    session_dir: Path = None,
    output_csv: Path = None,
    inv_iters: int = 20,
):
    if output_csv is None:
        output_csv = NOTEBOOK_DIR / "dist_gt_lesion_pred_lesion_vxm.csv"

    rows_out = []
    n = len(df)

    for idx in range(n):
        row = df.iloc[idx]
        pid   = row["patient_id"]
        study = row["study_id"]
        side  = row["breast_side"]

        dist_mm = np.nan
        try:
            res = compute_gt_lesion_pred_lesion_for_case(
                df=df,
                case_idx=idx,
                run_dir=run_dir,
                session_dir=session_dir,
                verbose=False,
                inv_iters=inv_iters,
            )
            dist_mm = res["dist_gt_lesion_pred_lesion_mm"]
            print(f"[{idx+1:03d}/{n}] pid={pid} study={study} side={side}  dist={dist_mm:.2f} mm")
        except Exception as e:
            print(f"[{idx+1:03d}/{n}] pid={pid} study={study} side={side}  FEHLER -> {repr(e)}")

        rows_out.append({
            "patient_id": pid,
            "study_id": study,
            "breast_side": side,
            "dist_gt_lesion_pred_lesion_mm": dist_mm,
        })

    df_out = pd.DataFrame(rows_out)
    df_out.to_csv(output_csv, index=False)
    print("\nFertig. CSV geschrieben nach:", output_csv)
    return df_out

In [9]:
# ============================================================
# Widgets / UI
# ============================================================

csv_text = widgets.Text(
    value=str(DEFAULT_TEST_CSV),
    description="CSV:",
    layout=widgets.Layout(width="88%"),
)

inf_base_text = widgets.Text(
    value=str(DEFAULT_INF_BASE_DIR),
    description="Inf base:",
    layout=widgets.Layout(width="88%"),
)

reload_button = widgets.Button(
    description="Neu laden",
    tooltip="CSV + Inference-Ordner scannen",
)

run_dropdown = widgets.Dropdown(
    options=[],
    description="Run:",
    layout=widgets.Layout(width="88%"),
)

session_dropdown = widgets.Dropdown(
    options=[],
    description="Session:",
    layout=widgets.Layout(width="88%"),
)

case_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description="Case:",
    continuous_update=False,
)

axis_dropdown = widgets.Dropdown(
    options=[("z (axial)", "z"), ("y (coronal)", "y"), ("x (sagittal)", "x")],
    value="z",
    description="Axis:",
)

patch_slider = widgets.IntSlider(
    value=41,
    min=0,
    max=121,
    step=2,
    description="Box:",
    continuous_update=False,
)

inv_iters_slider = widgets.IntSlider(
    value=20,
    min=1,
    max=60,
    step=1,
    description="Inv iters:",
    continuous_update=False,
)

single_button = widgets.Button(
    description="Single Case",
    button_style="info",
)

all_button = widgets.Button(
    description="Alle Fälle",
    button_style="success",
)

output_csv_text = widgets.Text(
    value=str(NOTEBOOK_DIR / "dist_gt_lesion_pred_lesion_vxm.csv"),
    description="Out CSV:",
    layout=widgets.Layout(width="88%"),
)

output_area = widgets.Output()

_df = None
_runs = []


def refresh_runs_and_sessions():
    global _df, _runs

    csv_path = Path(csv_text.value).expanduser()
    base_dir = Path(inf_base_text.value).expanduser()

    _df = load_pairs_csv(csv_path)
    case_slider.max = max(0, len(_df) - 1)
    case_slider.value = 0

    _runs = discover_runs(base_dir)
    if not _runs:
        run_dropdown.options = []
        session_dropdown.options = []
        raise FileNotFoundError(f"Keine Inference-Runs gefunden unter: {base_dir}")

    run_dropdown.options = [(r.name, str(r)) for r in _runs]
    run_dropdown.value = str(_runs[-1])

    refresh_sessions_for_run()
    print(f"[INFO] CSV geladen: {csv_path} ({len(_df)} Fälle)")
    print(f"[INFO] Inference base: {base_dir}")
    print(f"[INFO] Gefundene Runs: {len(_runs)}")


def refresh_sessions_for_run(*args):
    run_dir = Path(run_dropdown.value)
    sess = list_session_dirs(run_dir)

    if sess:
        session_dropdown.options = [(s.name, str(s)) for s in sess]
        session_dropdown.value = str(sess[-1])
    else:
        session_dropdown.options = [("(direkt im Run)", str(run_dir))]
        session_dropdown.value = str(run_dir)


def get_selected_run_and_session():
    run_dir = Path(run_dropdown.value)
    sess_dir = Path(session_dropdown.value)
    return run_dir, sess_dir


def on_reload_clicked(_):
    with output_area:
        clear_output()
        try:
            refresh_runs_and_sessions()
        except Exception as e:
            print("FEHLER beim Neu-Laden:")
            print(repr(e))


def on_run_change(change):
    if change["name"] == "value":
        try:
            refresh_sessions_for_run()
        except Exception as e:
            with output_area:
                clear_output()
                print("FEHLER beim Session-Refresh:")
                print(repr(e))


def on_single_clicked(_):
    with output_area:
        clear_output()
        try:
            if _df is None:
                refresh_runs_and_sessions()

            run_dir, sess_dir = get_selected_run_and_session()
            print("Run dir:", run_dir)
            print("Session dir:", sess_dir)
            print("Inv iters:", inv_iters_slider.value)

            res = compute_gt_lesion_pred_lesion_for_case(
                df=_df,
                case_idx=case_slider.value,
                run_dir=run_dir,
                session_dir=sess_dir,
                verbose=True,
                inv_iters=inv_iters_slider.value,
            )

            print("\nErgebnis:")
            print(f"  patient_id : {res['patient_id']}")
            print(f"  study_id   : {res['study_id']}")
            print(f"  breast_side: {res['breast_side']}")
            print(f"  Distanz    : {res['dist_gt_lesion_pred_lesion_mm']:.2f} mm")

            show_points_on_fixed_volume(
                fixed_path=res["fixed_needle_pre_path"],
                xyz_pred_mm=res["xyz_lesion_pred_mm"],
                xyz_gt_mm=res["xyz_lesion_gt_mm"],
                axis=axis_dropdown.value,
                patch=patch_slider.value,
                title=(
                    f"Case {case_slider.value}: "
                    f"{res['patient_id']}_{res['study_id']}_{res['breast_side']}"
                ),
                dist_mm=res["dist_gt_lesion_pred_lesion_mm"],
            )

        except Exception as e:
            print("FEHLER bei Single Case:")
            print(repr(e))


def on_all_clicked(_):
    with output_area:
        clear_output()
        try:
            if _df is None:
                refresh_runs_and_sessions()

            run_dir, sess_dir = get_selected_run_and_session()
            out_csv = Path(output_csv_text.value).expanduser()

            print("Run dir:", run_dir)
            print("Session dir:", sess_dir)
            print("Inv iters:", inv_iters_slider.value)
            print("Out CSV:", out_csv)

            df_out = run_all_cases(
                df=_df,
                run_dir=run_dir,
                session_dir=sess_dir,
                output_csv=out_csv,
                inv_iters=inv_iters_slider.value,
            )
            display(df_out.head())
        except Exception as e:
            print("FEHLER bei Alle-Fälle-Auswertung:")
            print(repr(e))


reload_button.on_click(on_reload_clicked)
run_dropdown.observe(on_run_change)
single_button.on_click(on_single_clicked)
all_button.on_click(on_all_clicked)

controls = widgets.VBox([
    csv_text,
    inf_base_text,
    widgets.HBox([reload_button]),
    run_dropdown,
    session_dropdown,
    case_slider,
    widgets.HBox([axis_dropdown, patch_slider, inv_iters_slider]),
    widgets.HBox([single_button, all_button]),
    output_csv_text,
])

display(controls, output_area)

# Initial laden
with output_area:
    clear_output()
    try:
        refresh_runs_and_sessions()
    except Exception as e:
        print("Initialer Load fehlgeschlagen:")
        print(repr(e))

Output()